In [ ]:
from sagemaker.workflow.function_step import step
from sagemaker.workflow.pipeline import Pipeline
import sagemaker
from sagemaker.workflow.parameters import ParameterInteger
from sagemaker.workflow.execution_variables import ExecutionVariables
import utils

In [ ]:
role = sagemaker.get_execution_role()
sts = boto3.client("sts")
account_id = sts.get_caller_identity()["Account"]

region = boto3.Session().region_name
print(f"Region: {region}")
user = "grupo5"

default_bucket = "s3-mlflow-artifacts-mlflow-tracking-server-grupo5-01"
default_prefix = f"sagemaker/clients-attrition/{user}"

default_path = default_bucket + "/" + default_prefix
athena_database = "glue-catalog-database-utec-bank-01"

#Pipeline configuration
instance_type = "ml.m5.large"
pipeline_name = f"pipeline-inference-{user}"
model_name = f"model-attrition-{user}"
model_version = "latest"
# cod_month = ParameterInteger(name="PeriodoCarga")

#MLFlow configuration
tracking_server_arn = f"arn:aws:sagemaker:{region}:{account_id}:mlflow-tracking-server/{TRACKING_SERVER}"
experiment_name = f"pipeline-inference-exp-{user}"

In [ ]:
@step(
    name="DataPull",
    instance_type=instance_type,
    dependencies="./data_pull_requirements.txt"
)
def data_pull(experiment_name: str, run_name: str,
              cod_month: int) -> tuple[str, str, str]:
    import awswrangler as wr
    import mlflow

    mlflow.set_tracking_uri(tracking_server_arn)
    mlflow.set_experiment(experiment_name)
    query = """
        SELECT  transaction_id
                ,amount
                ,merchant_category
                ,merchant_country
                ,card_present
                ,is_fraud
                ,cod_month
                ,trx_vel_last_1mths
                ,trx_vel_last_2mths
                ,amt_vel_last_1mths
                ,amt_vel_last_2mths
        FROM    RISK_MANAGEMENT.CREDIT_CARD_TRANSACTIONS
        WHERE   cod_month = {}
    """.format(cod_month)

    inf_raw_s3_path = f"s3://{default_path}/inf-raw-data/{cod_month}.csv"
    with mlflow.start_run(run_name=run_name) as run:
        run_id = run.info.run_id
        with mlflow.start_run(run_name="DataPull", nested=True):
            df = wr.athena.read_sql_query(sql=query, database="risk_management")
            df.to_csv(inf_raw_s3_path, index=False)
            mlflow.log_input(
                mlflow.data.from_pandas(df, inf_raw_s3_path),
                context="DataPull"
            )
    return inf_raw_s3_path, experiment_name, run_id

In [ ]:
@step(
    name="ModelInference",
    instance_type=instance_type,
    dependencies="./model_training_requirements.txt"
)
def model_inference(inf_raw_s3_path: str, experiment_name: str,
                    run_id: str, cod_month: int) -> tuple[str, str, str]:
    import pandas as pd
    import mlflow

    mlflow.set_tracking_uri(tracking_server_arn)
    mlflow.set_experiment(experiment_name)
    FEATURES = ['card_present', 'trx_vel_last_1mths', 'trx_vel_last_2mths',
                'amt_vel_last_1mths', 'amt_vel_last_2mths']
    model_uri = f"models:/{model_name}/{model_version}"
    df = pd.read_csv(inf_raw_s3_path)
    X = df[FEATURES]
    model = mlflow.xgboost.load_model(model_uri)
    df["prob"] = model.predict_proba(X)[:, 1]
    inf_proc_s3_path = f"s3://{default_path}/inf-proc-data/{cod_month}.csv"

    with mlflow.start_run(run_id=run_id):
        with mlflow.start_run(run_name="ModelInference", nested=True):
            df.to_csv(inf_proc_s3_path, index=False)
            mlflow.log_input(
                mlflow.data.from_pandas(df, inf_proc_s3_path),
                context="ModelInference"
            )
    return inf_proc_s3_path, experiment_name, run_id